# 📊 Evaluation, Benchmarking, and Ethics

This final notebook covers comprehensive evaluation techniques, benchmarking methodologies, and ethical considerations for LLM classification systems.

## 🎯 Learning Objectives

By the end of this notebook, you will:
1. Master comprehensive evaluation metrics for LLMs
2. Implement robust benchmarking frameworks
3. Understand model calibration and uncertainty estimation
4. Detect and mitigate bias in LLM outputs
5. Ensure fairness across different demographic groups
6. Implement responsible AI practices
7. Create comprehensive evaluation pipelines
8. Understand regulatory compliance and best practices

## 🔧 Prerequisites

- Completed Notebooks 1-4 (Fundamentals, vLLM, Fine-tuning, Production)
- Understanding of machine learning evaluation metrics
- Familiarity with statistical analysis
- Basic knowledge of AI ethics and fairness

In [1]:
# These packages are used by this notebook but may not be
# pre-installed in the lab environment.
!pip install -q evaluate rouge-score bert-score

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8): No such file or directory

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, pipeline
)
import evaluate
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)
from typing import Dict, List, Any, Tuple
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print("📊 Evaluation & Ethics Environment Ready!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

📊 Evaluation & Ethics Environment Ready!
PyTorch version: 2.10.0+cpu
CUDA available: False


In [3]:
# Load a pre-trained model and tokenizer for demonstration
# Using a small model for faster execution
model_name = "distilbert/distilroberta-base" # Example model name
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(f"✅ Model '{model_name}' and tokenizer loaded!")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model 'distilbert/distilroberta-base' and tokenizer loaded!


## 📈 Comprehensive Evaluation Metrics

Understanding and implementing comprehensive evaluation metrics for LLM classification.

In [4]:
# Comprehensive evaluation framework
class LLMEvaluator:
    """Comprehensive evaluator for LLM classification systems"""

    def __init__(self):
        self.metrics = {}
        self.setup_metrics()

    def setup_metrics(self):
        """Setup all evaluation metrics"""
        # Standard classification metrics
        self.metrics['accuracy'] = evaluate.load('accuracy')
        self.metrics['precision'] = evaluate.load('precision')
        self.metrics['recall'] = evaluate.load('recall')
        self.metrics['f1'] = evaluate.load('f1')

        # Text quality metrics
        try:
            self.metrics['rouge'] = evaluate.load('rouge')
        except:
            print("ROUGE metric not available")

        try:
            self.metrics['bertscore'] = evaluate.load('bertscore')
        except:
            print("BERTScore metric not available")

    def evaluate_classification(self, predictions, references, labels=None):
        """Evaluate classification performance"""

        results = {}

        # Basic metrics
        results['accuracy'] = self.metrics['accuracy'].compute(
            predictions=predictions, references=references
        )

        results['precision'] = self.metrics['precision'].compute(
            predictions=predictions, references=references, average='weighted'
        )

        results['recall'] = self.metrics['recall'].compute(
            predictions=predictions, references=references, average='weighted'
        )

        results['f1'] = self.metrics['f1'].compute(
            predictions=predictions, references=references, average='weighted'
        )

        # Per-class metrics
        if labels:
            per_class = precision_recall_fscore_support(
                references, predictions, average=None, labels=labels
            )
            results['per_class'] = {
                'precision': dict(zip(labels, per_class[0])),
                'recall': dict(zip(labels, per_class[1])),
                'f1': dict(zip(labels, per_class[2]))
            }

        # Confusion matrix
        results['confusion_matrix'] = confusion_matrix(references, predictions)

        return results

    def evaluate_text_quality(self, predictions, references):
        """Evaluate text quality metrics"""

        results = {}

        # ROUGE scores
        if 'rouge' in self.metrics:
            rouge_results = self.metrics['rouge'].compute(
                predictions=predictions, references=references
            )
            results['rouge'] = rouge_results

        # BERTScore
        if 'bertscore' in self.metrics:
            bert_results = self.metrics['bertscore'].compute(
                predictions=predictions, references=references, lang='en'
            )
            results['bertscore'] = {
                'precision': np.mean(bert_results['precision']),
                'recall': np.mean(bert_results['recall']),
                'f1': np.mean(bert_results['f1'])
            }

        return results

    def evaluate_calibration(self, predictions, confidences, references):
        """Evaluate model calibration and uncertainty"""

        # Brier score for calibration
        brier_score = np.mean((confidences - references) ** 2)

        # Expected calibration error
        n_bins = 10
        bins = np.linspace(0, 1, n_bins + 1)
        bin_indices = np.digitize(confidences, bins) - 1

        ece = 0
        for i in range(n_bins):
            mask = bin_indices == i
            if np.sum(mask) > 0:
                bin_conf = np.mean(confidences[mask])
                bin_acc = np.mean(predictions[mask] == references[mask])
                ece += np.sum(mask) * np.abs(bin_conf - bin_acc)
        ece /= len(predictions)

        return {
            'brier_score': brier_score,
            'expected_calibration_error': ece
        }

# Initialize evaluator
evaluator = LLMEvaluator()
print("✅ Comprehensive LLM Evaluator initialized!")

# Test with sample data
sample_predictions = [0, 1, 1, 0, 1]
sample_references = [0, 1, 0, 0, 1]
sample_labels = [0, 1]

eval_results = evaluator.evaluate_classification(
    sample_predictions, sample_references, sample_labels
)

print("\n📊 Sample Evaluation Results:")
for metric, value in eval_results.items():
    if metric != 'confusion_matrix':
        print(f"   {metric}: {value}")
    else:
        print(f"   {metric}:\n{value}")

✅ Comprehensive LLM Evaluator initialized!

📊 Sample Evaluation Results:
   accuracy: {'accuracy': 0.8}
   precision: {'precision': 0.8666666666666666}
   recall: {'recall': 0.8}
   f1: {'f1': 0.8}
   per_class: {'precision': {0: np.float64(1.0), 1: np.float64(0.6666666666666666)}, 'recall': {0: np.float64(0.6666666666666666), 1: np.float64(1.0)}, 'f1': {0: np.float64(0.8), 1: np.float64(0.8)}}
   confusion_matrix:
[[2 1]
 [0 2]]


## 🏁 Benchmarking Framework

Creating a comprehensive benchmarking framework for LLM classification systems.

In [5]:
# Comprehensive benchmarking framework
class LLMBenchmark:
    """Comprehensive benchmarking framework for LLM classification"""

    def __init__(self):
        self.benchmarks = {}
        self.setup_benchmarks()

    def setup_benchmarks(self):
        """Setup standard benchmarks"""

        # Standard classification benchmarks
        self.benchmarks = {
            'sentiment_analysis': {
                'dataset': 'imdb',
                'task': 'binary_classification',
                'metrics': ['accuracy', 'f1', 'precision', 'recall']
            },
            'topic_classification': {
                'dataset': 'ag_news',
                'task': 'multiclass_classification',
                'metrics': ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']
            },
            'emotion_detection': {
                'dataset': 'emotion',
                'task': 'multiclass_classification',
                'metrics': ['accuracy', 'f1_macro']
            },
            'intent_classification': {
                'dataset': 'banking77',
                'task': 'multiclass_classification',
                'metrics': ['accuracy', 'f1_macro']
            }
        }

    def run_benchmark(self, model, tokenizer, benchmark_name: str, sample_size: int = 100):
        """Run a specific benchmark"""

        if benchmark_name not in self.benchmarks:
            raise ValueError(f"Benchmark {benchmark_name} not found")

        benchmark_config = self.benchmarks[benchmark_name]

        print(f"🏁 Running {benchmark_name} benchmark...")
        print(f"📊 Dataset: {benchmark_config['dataset']}")
        print(f"🎯 Task: {benchmark_config['task']}")

        # Load dataset
        try:
            dataset = load_dataset(benchmark_config['dataset'], split=f'test[:{sample_size}]')
        except:
            print(f"Could not load {benchmark_config['dataset']}, using IMDB as fallback")
            dataset = load_dataset('imdb', split=f'test[:{sample_size}]')

        # Prepare data for classification
        if benchmark_name == 'sentiment_analysis':
            texts = dataset['text']
            labels = dataset['label']
            label_names = ['negative', 'positive']
        else:
            # For other benchmarks, use first sample_size examples
            texts = [str(x) for x in dataset[:sample_size]['text']]
            labels = list(range(len(texts)))  # Dummy labels for demo
            label_names = ['class_' + str(i) for i in range(len(set(labels)))]

        # Run inference
        predictions = []
        confidences = []

        model.eval()
        for text in texts[:min(20, len(texts))]:  # Limit for demo
            prompt = f"Classify the sentiment of this text: {text[:200]}...\nSentiment:"

            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=10,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

            generated = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

            # Simple classification logic
            if 'positive' in generated.lower():
                pred = 1
            elif 'negative' in generated.lower():
                pred = 0
            else:
                pred = 0  # Default

            predictions.append(pred)
            confidences.append(0.8)  # Dummy confidence

        # Evaluate
        true_labels = labels[:len(predictions)]
        results = evaluator.evaluate_classification(predictions, true_labels)

        # Add benchmark metadata
        results['benchmark'] = benchmark_name
        results['dataset'] = benchmark_config['dataset']
        results['sample_size'] = len(predictions)

        print(f"✅ Benchmark completed: {len(predictions)} samples evaluated")
        return results

    def run_all_benchmarks(self, model, tokenizer, sample_size: int = 50):
        """Run all available benchmarks"""

        all_results = {}

        for benchmark_name in self.benchmarks.keys():
            try:
                results = self.run_benchmark(model, tokenizer, benchmark_name, sample_size)
                all_results[benchmark_name] = results
            except Exception as e:
                print(f"❌ Benchmark {benchmark_name} failed: {e}")
                all_results[benchmark_name] = {'error': str(e)}

        return all_results

# Initialize benchmark framework
benchmark = LLMBenchmark()
print("✅ LLM Benchmarking Framework initialized!")
print(f"📊 Available benchmarks: {list(benchmark.benchmarks.keys())}")

✅ LLM Benchmarking Framework initialized!
📊 Available benchmarks: ['sentiment_analysis', 'topic_classification', 'emotion_detection', 'intent_classification']


## 🎯 Bias Detection and Fairness Analysis

Implementing comprehensive bias detection and fairness analysis for LLM systems.

In [6]:
# Bias detection and fairness analysis
class BiasDetector:
    """Comprehensive bias detection for LLM classification systems"""

    def __init__(self):
        self.bias_metrics = {}

    def detect_demographic_bias(self, predictions, true_labels, demographics):
        """Detect bias across demographic groups"""

        results = {}

        for demo_group, demo_labels in demographics.items():
            group_predictions = [pred for pred, demo in zip(predictions, demo_labels) if demo == 1]
            group_true = [true for true, demo in zip(true_labels, demo_labels) if demo == 1]

            if len(group_predictions) > 10:  # Minimum sample size
                accuracy = accuracy_score(group_true, group_predictions)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    group_true, group_predictions, average='weighted'
                )

                results[demo_group] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'sample_size': len(group_predictions)
                }

        # Calculate fairness metrics
        if len(results) > 1:
            accuracies = [metrics['accuracy'] for metrics in results.values()]
            fairness_score = 1 - (np.std(accuracies) / np.mean(accuracies))
            results['fairness_score'] = fairness_score

            # Statistical parity difference
            positive_rates = [sum(1 for p in group_predictions if p == 1) / len(group_predictions)
                            for group_predictions in [group_predictions for group_predictions in results.values() if isinstance(group_predictions, dict)]]
            if len(positive_rates) > 1:
                results['statistical_parity_diff'] = max(positive_rates) - min(positive_rates)

        return results

    def detect_text_bias(self, texts, predictions):
        """Detect bias in text content and predictions"""

        bias_indicators = {
            'gender_bias': ['he', 'she', 'man', 'woman', 'male', 'female'],
            'racial_bias': ['race', 'ethnic', 'minority', 'diverse', 'cultural'],
            'socioeconomic_bias': ['rich', 'poor', 'wealthy', 'poverty', 'class'],
            'age_bias': ['young', 'old', 'elderly', 'youth', 'senior']
        }

        bias_analysis = {}

        for bias_type, indicators in bias_indicators.items():
            indicator_counts = []
            prediction_distribution = {0: 0, 1: 0}

            for text, pred in zip(texts, predictions):
                text_lower = text.lower()
                count = sum(1 for indicator in indicators if indicator in text_lower)
                indicator_counts.append(count)
                prediction_distribution[pred] += 1

            # Analyze correlation between bias indicators and predictions
            bias_score = np.corrcoef(indicator_counts, predictions)[0, 1] if len(set(indicator_counts)) > 1 else 0

            bias_analysis[bias_type] = {
                'correlation': bias_score,
                'avg_indicators': np.mean(indicator_counts),
                'prediction_distribution': prediction_distribution
            }

        return bias_analysis

    def generate_bias_report(self, demographic_bias, text_bias):
        """Generate comprehensive bias report"""

        report = {
            'summary': {
                'demographic_groups_analyzed': len([k for k in demographic_bias.keys() if k != 'fairness_score']),
                'fairness_score': demographic_bias.get('fairness_score', 'N/A'),
                'statistical_parity_diff': demographic_bias.get('statistical_parity_diff', 'N/A')
            },
            'demographic_bias': demographic_bias,
            'text_bias': text_bias,
            'recommendations': self._generate_recommendations(demographic_bias, text_bias)
        }

        return report

    def _generate_recommendations(self, demographic_bias, text_bias):
        """Generate mitigation recommendations"""

        recommendations = []

        # Demographic fairness recommendations
        fairness_score = demographic_bias.get('fairness_score', 1)
        if fairness_score < 0.8:
            recommendations.append("Implement fairness constraints during training")
            recommendations.append("Use reweighting or resampling techniques")
            recommendations.append("Consider adversarial debiasing methods")

        # Text bias recommendations
        for bias_type, analysis in text_bias.items():
            if abs(analysis['correlation']) > 0.3:
                recommendations.append(f"Address {bias_type.replace('_', ' ')} in training data")
                recommendations.append(f"Implement bias detection preprocessing for {bias_type}")

        if not recommendations:
            recommendations.append("No significant bias detected - continue monitoring")

        return recommendations

# Initialize bias detector
bias_detector = BiasDetector()
print("✅ Bias Detection Framework initialized!")

# Example bias analysis
sample_texts = [
    "The young man worked hard",
    "The elderly woman was wise",
    "Rich people have advantages",
    "Poor communities need help"
]

sample_predictions = [1, 1, 0, 1]  # Dummy predictions
text_bias = bias_detector.detect_text_bias(sample_texts, sample_predictions)

print("\n🔍 Sample Text Bias Analysis:")
for bias_type, analysis in text_bias.items():
    print(f"   {bias_type}: correlation = {analysis['correlation']:.3f}")

✅ Bias Detection Framework initialized!

🔍 Sample Text Bias Analysis:
   gender_bias: correlation = 0.775
   racial_bias: correlation = 0.000
   socioeconomic_bias: correlation = -0.577
   age_bias: correlation = 0.577


## 📊 Model Calibration and Uncertainty

Understanding and improving model calibration and uncertainty estimation.

In [7]:
# Model calibration and uncertainty estimation
class UncertaintyEstimator:
    """Comprehensive uncertainty estimation for LLM predictions"""

    def __init__(self):
        self.calibration_methods = {}

    def estimate_uncertainty(self, model, tokenizer, text: str, num_samples: int = 10):
        """Estimate prediction uncertainty using MC Dropout.

        Enables dropout at inference time and runs multiple forward passes
        to estimate how confident the model is in its prediction.
        """
        predictions = []
        probabilities = []

        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}

        # Enable dropout for MC Dropout uncertainty estimation
        model.train()

        for _ in range(num_samples):
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                pred = torch.argmax(probs, dim=-1).item()
                predictions.append(pred)
                probabilities.append(probs[0].cpu().numpy())

        # Restore eval mode
        model.eval()

        # Calculate uncertainty metrics
        pred_array = np.array(predictions)
        prob_array = np.array(probabilities)
        mean_probs = np.mean(prob_array, axis=0)
        mean_prediction = np.mean(pred_array)
        prediction_std = np.std(pred_array)
        entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10))
        confidence = 1 - prediction_std

        return {
            'mean_prediction': mean_prediction,
            'prediction_std': prediction_std,
            'entropy': entropy,
            'confidence': confidence,
            'mean_probabilities': mean_probs.tolist(),
            'prediction_distribution': {
                'class_0': int(np.sum(pred_array == 0)),
                'class_1': int(np.sum(pred_array == 1))
            }
        }

    def calibrate_model(self, predictions, confidences, true_labels):
        """Calibrate model predictions using temperature scaling"""

        def temperature_scale(confidence, temperature=1.5):
            return 1 / (1 + np.exp(-(confidence - 0.5) / temperature))

        calibrated_confidences = [temperature_scale(c) for c in confidences]

        ece_before = self.expected_calibration_error(confidences, predictions, true_labels)
        ece_after = self.expected_calibration_error(calibrated_confidences, predictions, true_labels)

        return {
            'calibrated_confidences': calibrated_confidences,
            'ece_before': ece_before,
            'ece_after': ece_after,
            'improvement': ece_before - ece_after
        }

    def expected_calibration_error(self, confidences, predictions, true_labels, n_bins=10):
        """Calculate Expected Calibration Error (ECE)"""

        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        bin_indices = np.digitize(confidences, bin_boundaries) - 1

        ece = 0
        for i in range(n_bins):
            mask = bin_indices == i
            if np.sum(mask) > 0:
                bin_conf = np.mean([c for c, m in zip(confidences, mask) if m])
                bin_acc = np.mean([p == t for p, t, m in zip(predictions, true_labels, mask) if m])
                ece += np.sum(mask) * abs(bin_conf - bin_acc)

        return ece / len(predictions)

    def plot_calibration_curve(self, confidences, predictions, true_labels):
        """Plot calibration curve to visualize model calibration"""

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')

        prob_true, prob_pred = calibration_curve(true_labels, confidences, n_bins=10)
        ax.plot(prob_pred, prob_true, 'ro-', label='Model Calibration')

        ax.set_xlabel('Predicted Probability')
        ax.set_ylabel('True Probability')
        ax.set_title('Calibration Curve')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()

# Initialize uncertainty estimator
uncertainty_estimator = UncertaintyEstimator()
print("\u2705 Uncertainty Estimation Framework initialized!")

# Example uncertainty estimation
test_text = "This product is amazing! Highly recommended."

print(f"\n\U0001f50d Analyzing uncertainty for: '{test_text}'")
uncertainty = uncertainty_estimator.estimate_uncertainty(
    model, tokenizer, test_text, num_samples=5
)

print("\U0001f4ca Uncertainty Analysis:")
print(f"   Mean prediction: {uncertainty['mean_prediction']:.3f}")
print(f"   Prediction std: {uncertainty['prediction_std']:.3f}")
print(f"   Confidence: {uncertainty['confidence']:.3f}")
print(f"   Distribution: {uncertainty['prediction_distribution']}")


✅ Uncertainty Estimation Framework initialized!

🔍 Analyzing uncertainty for: 'This product is amazing! Highly recommended.'
📊 Uncertainty Analysis:
   Mean prediction: 0.000
   Prediction std: 0.000
   Confidence: 1.000
   Distribution: {'class_0': 5, 'class_1': 0}


## ⚖️ Ethical AI and Responsible Deployment

Implementing ethical guidelines and responsible AI practices for LLM classification.

In [8]:
# Ethical AI framework
class EthicalAIGuardian:
    """Comprehensive ethical AI framework for LLM classification"""

    def __init__(self):
        self.ethical_checks = {}
        self.setup_ethical_checks()

    def setup_ethical_checks(self):
        """Setup comprehensive ethical checks"""

        self.ethical_checks = {
            'harmful_content': {
                'keywords': ['harm', 'violence', 'abuse', 'illegal', 'dangerous'],
                'severity': 'high',
                'action': 'block'
            },
            'privacy_violation': {
                'keywords': ['personal', 'private', 'confidential', 'sensitive'],
                'severity': 'high',
                'action': 'flag'
            },
            'bias_indicators': {
                'keywords': ['stereotype', 'discrimination', 'bias', 'prejudice'],
                'severity': 'medium',
                'action': 'warn'
            },
            'misinformation': {
                'keywords': ['false', 'misleading', 'inaccurate', 'fake'],
                'severity': 'medium',
                'action': 'flag'
            }
        }

    def check_input_ethics(self, text: str) -> Dict[str, Any]:
        """Check input text for ethical concerns"""

        text_lower = text.lower()
        concerns = []

        for check_name, check_config in self.ethical_checks.items():
            for keyword in check_config['keywords']:
                if keyword in text_lower:
                    concerns.append({
                        'type': check_name,
                        'keyword': keyword,
                        'severity': check_config['severity'],
                        'action': check_config['action']
                    })
                    break

        # Determine overall risk level
        if any(c['severity'] == 'high' for c in concerns):
            risk_level = 'high'
            recommended_action = 'block'
        elif concerns:
            risk_level = 'medium'
            recommended_action = 'flag'
        else:
            risk_level = 'low'
            recommended_action = 'allow'

        return {
            'risk_level': risk_level,
            'concerns': concerns,
            'recommended_action': recommended_action,
            'allow_processing': risk_level == 'low'
        }

    def check_output_ethics(self, output: str) -> Dict[str, Any]:
        """Check model output for ethical concerns"""

        # Similar to input checking but with different focus
        output_lower = output.lower()

        # Check for potentially harmful outputs
        harmful_patterns = [
            'harm yourself',
            'illegal activities',
            'discriminatory content',
            'misleading information'
        ]

        detected_harm = []
        for pattern in harmful_patterns:
            if pattern in output_lower:
                detected_harm.append(pattern)

        return {
            'harmful_content': detected_harm,
            'needs_review': len(detected_harm) > 0,
            'safe_for_release': len(detected_harm) == 0
        }

    def generate_ethics_report(self, inputs: List[str], outputs: List[str]) -> Dict[str, Any]:
        """Generate comprehensive ethics report"""

        input_checks = [self.check_input_ethics(text) for text in inputs]
        output_checks = [self.check_output_ethics(output) for output in outputs]

        # Aggregate statistics
        high_risk_inputs = sum(1 for check in input_checks if check['risk_level'] == 'high')
        flagged_outputs = sum(1 for check in output_checks if check['needs_review'])

        # Most common concerns
        all_concerns = []
        for check in input_checks:
            all_concerns.extend([c['type'] for c in check['concerns']])

        concern_counts = {}
        for concern in all_concerns:
            concern_counts[concern] = concern_counts.get(concern, 0) + 1

        return {
            'total_samples': len(inputs),
            'high_risk_inputs': high_risk_inputs,
            'flagged_outputs': flagged_outputs,
            'risk_percentage': (high_risk_inputs / len(inputs)) * 100,
            'common_concerns': concern_counts,
            'recommendations': self._generate_ethics_recommendations(high_risk_inputs, len(inputs))
        }

    def _generate_ethics_recommendations(self, high_risk_count: int, total_count: int) -> List[str]:
        """Generate ethics recommendations based on analysis"""

        risk_percentage = (high_risk_count / total_count) * 100

        recommendations = []

        if risk_percentage > 20:
            recommendations.extend([
                "Implement stricter input filtering",
                "Add human review for high-risk content",
                "Consider model fine-tuning for safer outputs"
            ])
        elif risk_percentage > 10:
            recommendations.extend([
                "Enhance content moderation",
                "Add warning labels for potentially sensitive content",
                "Implement usage logging and monitoring"
            ])
        else:
            recommendations.append("Current ethical safeguards are adequate")

        recommendations.extend([
            "Regular ethics audits and bias testing",
            "Transparent documentation of limitations",
            "User feedback mechanisms for ethical concerns"
        ])

        return recommendations

# Initialize ethical AI guardian
ethics_guardian = EthicalAIGuardian()
print("🛡️ Ethical AI Guardian initialized!")

# Test ethical checking
test_inputs = [
    "How can I improve my productivity?",
    "Tell me about harmful activities",
    "What's the weather like?",
    "Share private information"
]

print("\n🔍 Ethical Analysis of Sample Inputs:")
for i, text in enumerate(test_inputs):
    check = ethics_guardian.check_input_ethics(text)
    print(f"   Input {i+1}: {text[:30]}... -> Risk: {check['risk_level']}")

🛡️ Ethical AI Guardian initialized!

🔍 Ethical Analysis of Sample Inputs:
   Input 1: How can I improve my productiv... -> Risk: low
   Input 2: Tell me about harmful activiti... -> Risk: high
   Input 3: What's the weather like?... -> Risk: low
   Input 4: Share private information... -> Risk: high


## 📈 Regulatory Compliance and Best Practices

Understanding regulatory requirements and implementing compliance measures.

In [9]:
# Regulatory compliance framework
class ComplianceManager:
    """Comprehensive regulatory compliance for LLM systems"""

    def __init__(self):
        self.regulations = {}
        self.setup_regulations()

    def setup_regulations(self):
        """Setup regulatory frameworks and requirements"""

        self.regulations = {
            'GDPR': {
                'scope': 'EU data protection',
                'requirements': [
                    'Data minimization',
                    'Purpose limitation',
                    'Consent management',
                    'Right to erasure',
                    'Data portability'
                ],
                'relevance': 'High for EU users'
            },
            'CCPA': {
                'scope': 'California privacy',
                'requirements': [
                    'Right to know',
                    'Right to delete',
                    'Right to opt-out',
                    'Non-discrimination'
                ],
                'relevance': 'High for US users'
            },
            'AI_Act': {
                'scope': 'EU AI regulation',
                'requirements': [
                    'Risk assessment',
                    'Transparency',
                    'Human oversight',
                    'Data governance',
                    'Accuracy requirements'
                ],
                'relevance': 'Critical for high-risk AI systems'
            },
            'Industry_Standards': {
                'scope': 'General AI best practices',
                'requirements': [
                    'Bias auditing',
                    'Explainability',
                    'Robustness testing',
                    'Incident response',
                    'Regular updates'
                ],
                'relevance': 'Essential for all deployments'
            }
        }

    def assess_compliance(self, system_config: Dict[str, Any]) -> Dict[str, Any]:
        """Assess system compliance with regulations"""

        compliance_report = {}

        for reg_name, reg_info in self.regulations.items():
            compliance_score = self._calculate_compliance_score(system_config, reg_info)
            compliance_report[reg_name] = {
                'score': compliance_score,
                'status': 'compliant' if compliance_score >= 0.8 else 'needs_attention',
                'requirements': reg_info['requirements'],
                'gaps': self._identify_compliance_gaps(system_config, reg_info)
            }

        # Overall compliance
        avg_score = np.mean([report['score'] for report in compliance_report.values()])
        overall_status = 'compliant' if avg_score >= 0.8 else 'needs_improvement'

        return {
            'overall_score': avg_score,
            'overall_status': overall_status,
            'regulatory_compliance': compliance_report,
            'recommendations': self._generate_compliance_recommendations(compliance_report)
        }

    def _calculate_compliance_score(self, config: Dict[str, Any], regulation: Dict[str, Any]) -> float:
        """Calculate compliance score for a regulation (simplified)"""

        # Simplified scoring based on common compliance indicators
        score = 0.5  # Base score

        if config.get('has_monitoring', False):
            score += 0.1
        if config.get('has_bias_audits', False):
            score += 0.1
        if config.get('has_data_protection', False):
            score += 0.1
        if config.get('has_transparency', False):
            score += 0.1
        if config.get('has_incident_response', False):
            score += 0.1

        return min(score, 1.0)

    def _identify_compliance_gaps(self, config: Dict[str, Any], regulation: Dict[str, Any]) -> List[str]:
        """Identify compliance gaps"""

        gaps = []

        if not config.get('has_monitoring', False):
            gaps.append("Implement comprehensive monitoring")
        if not config.get('has_bias_audits', False):
            gaps.append("Conduct regular bias audits")
        if not config.get('has_data_protection', False):
            gaps.append("Enhance data protection measures")

        return gaps

    def _generate_compliance_recommendations(self, compliance_report: Dict[str, Any]) -> List[str]:
        """Generate compliance improvement recommendations"""

        recommendations = []

        for reg_name, report in compliance_report.items():
            if report['score'] < 0.8:
                recommendations.append(f"Improve {reg_name} compliance (current: {report['score']:.2f})")
                recommendations.extend(report['gaps'][:2])  # Top 2 gaps

        if not recommendations:
            recommendations.append("Compliance measures are adequate - continue monitoring")

        return recommendations

# Initialize compliance manager
compliance_manager = ComplianceManager()
print("📋 Compliance Manager initialized!")

# Test compliance assessment
sample_config = {
    'has_monitoring': True,
    'has_bias_audits': False,
    'has_data_protection': True,
    'has_transparency': False,
    'has_incident_response': True
}

compliance_report = compliance_manager.assess_compliance(sample_config)
print(f"\n📊 Compliance Assessment:")
print(f"   Overall Score: {compliance_report['overall_score']:.3f}")
print(f"   Status: {compliance_report['overall_status']}")
print(f"   Key Recommendations: {compliance_report['recommendations'][:2]}")

📋 Compliance Manager initialized!

📊 Compliance Assessment:
   Overall Score: 0.800
   Status: needs_improvement
   Key Recommendations: ['Improve GDPR compliance (current: 0.80)', 'Conduct regular bias audits']


## 📚 Key Takeaways and Final Summary

Congratulations! You've completed the comprehensive LLM Classification course covering evaluation, benchmarking, and ethics.

In [10]:
# Final comprehensive summary
def generate_course_summary():
    """Generate comprehensive course summary"""

    summary = {
        "course_structure": {
            "Notebook 1": "LLM Fundamentals and Architectures",
            "Notebook 2": "Efficient Inference with vLLM",
            "Notebook 3": "Advanced Fine-tuning with Unsloth",
            "Notebook 4": "Production Deployment and Scaling",
            "Notebook 5": "Evaluation, Benchmarking, and Ethics"
        },
        "key_achievements": [
            "Mastered modern LLM architectures and scaling laws",
            "Implemented efficient inference with vLLM optimizations",
            "Applied advanced fine-tuning with LoRA and QLoRA",
            "Built production-ready services with FastAPI",
            "Created comprehensive evaluation and benchmarking frameworks",
            "Implemented bias detection and fairness analysis",
            "Developed ethical AI practices and regulatory compliance",
            "Mastered model calibration and uncertainty estimation"
        ],
        "technical_mastery": {
            "Architectures": ["Transformer", "GPT-style", "BERT-style", "Encoder-Decoder"],
            "Techniques": ["LoRA", "QLoRA", "Quantization", "Continuous Batching"],
            "Tools": ["vLLM", "Unsloth", "FastAPI", "Evaluation Frameworks"],
            "Skills": ["Bias Detection", "Fairness Analysis", "Ethical AI", "Compliance"]
        },
        "production_readiness": {
            "Deployment": "Containerization, Orchestration, Auto-scaling",
            "Monitoring": "Performance Metrics, Health Checks, Alerting",
            "Security": "Input Validation, Rate Limiting, Authentication",
            "Compliance": "GDPR, CCPA, AI Act, Industry Standards"
        },
        "next_steps": [
            "Deploy your first production LLM service",
            "Implement comprehensive monitoring and alerting",
            "Conduct regular bias audits and ethical reviews",
            "Explore advanced architectures (Mixture of Experts, etc.)",
            "Contribute to open-source LLM projects",
            "Pursue LLM specialization certifications"
        ]
    }

    return summary

# Generate and display course summary
course_summary = generate_course_summary()

print("🎓 LLM CLASSIFICATION COURSE - FINAL SUMMARY")
print("=" * 60)

print("\n📚 Course Structure:")
for notebook, topic in course_summary['course_structure'].items():
    print(f"   {notebook}: {topic}")

print("\n🏆 Key Achievements:")
for i, achievement in enumerate(course_summary['key_achievements'], 1):
    print(f"   {i}. {achievement}")

print("\n🛠️ Technical Mastery:")
for category, items in course_summary['technical_mastery'].items():
    print(f"   {category}: {', '.join(items)}")

print("\n🚀 Production Readiness:")
for aspect, details in course_summary['production_readiness'].items():
    print(f"   {aspect}: {details}")

print("\n🔮 Next Steps:")
for i, step in enumerate(course_summary['next_steps'], 1):
    print(f"   {i}. {step}")

print("\n🎉 CONGRATULATIONS! You are now an LLM Classification expert!")
print("   Ready to build, deploy, and maintain production LLM systems.")
print("=" * 60)

🎓 LLM CLASSIFICATION COURSE - FINAL SUMMARY

📚 Course Structure:
   Notebook 1: LLM Fundamentals and Architectures
   Notebook 2: Efficient Inference with vLLM
   Notebook 3: Advanced Fine-tuning with Unsloth
   Notebook 4: Production Deployment and Scaling
   Notebook 5: Evaluation, Benchmarking, and Ethics

🏆 Key Achievements:
   1. Mastered modern LLM architectures and scaling laws
   2. Implemented efficient inference with vLLM optimizations
   3. Applied advanced fine-tuning with LoRA and QLoRA
   4. Built production-ready services with FastAPI
   5. Created comprehensive evaluation and benchmarking frameworks
   6. Implemented bias detection and fairness analysis
   7. Developed ethical AI practices and regulatory compliance
   8. Mastered model calibration and uncertainty estimation

🛠️ Technical Mastery:
   Architectures: Transformer, GPT-style, BERT-style, Encoder-Decoder
   Techniques: LoRA, QLoRA, Quantization, Continuous Batching
   Tools: vLLM, Unsloth, FastAPI, Evaluation